# 11 — libinjection ilə müqayisə

**libinjection** — ModSecurity-nin `@detectSQLi` operatorunun və bir çox WAF-ın istifadə etdiyi, tokenizasiya + "fingerprint" əsaslı sənaye SQLi detektoru.

**Versiya və binding:**
- Rəsmi C kitabxanası `libinjection/libinjection`, tag **v4.0.1** (GitHub tarball SHA-256 `tools/build_libinjection.py`-də sabitlənib).
- Yalnız `libinjection_sqli.c` paylaşılan kitabxana kimi `zig cc` ilə kompilyasiya olunur (`ziglang` 0.17.1, pip paketi; Windows-da sistem kompilyatoru lazım deyil).
- Python-dan `ctypes` ilə çağırılır: `src/libinj.py` (giriş UTF-8 bayt kimi, `libinjection_sqli(s, len, fingerprint)`; nəticə `1` → zərərli).
- PyPI-dəki `libinjection-python` / `pylibinjection` paketlərinin Windows + Python 3.13 üçün binary wheel-i olmadığı üçün istifadə edilmədi.
- ModSecurity CRS-in əlavə transformasiyaları (`t:urlDecodeUni` və s.) tətbiq olunmur; detektor eyni girişi görür: ssenari A-da bir dəfə decode olunmuş dəyər.

Qurmaq: `python tools/build_libinjection.py`.

Eyni dəstlər (seed 42 bölgüsü): orijinal test, bütün obfuskasiyalar (ssenari A və B), HttpParamsDataset (sızmasız), MySQL-də effektiv formalar.

In [1]:
import re, sys
from pathlib import Path
from urllib.parse import unquote

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

SEED = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import libinj
import obfuscation as obf
from blacklist import flags, sanitize_input, sanitize_search

PROC, RES, MODELS = ROOT / "data/processed", ROOT / "results", ROOT / "models"
print("libinjection", libinj.VERSION)

read = lambda n: pd.read_csv(PROC / n, keep_default_na=False, dtype={"Query": str, "Label": int})
train, val, test = read("train_v2.csv"), read("val_v2.csv"), read("test_v2.csv")
obf_long = pd.read_csv(PROC / "obfuscated_test_v2.csv", keep_default_na=False)
TECH_ORDER = list(obf.TECHNIQUES) + ["combination"]
VALID_TECHS = [t for t in TECH_ORDER if t not in obf.SYNTACTICALLY_INVALID]

key = lambda s: re.sub(r"\s+", " ", str(s)).strip().lower()
hp = pd.read_csv(ROOT / "data/raw/external/httpparams/payload_full.csv")
hp = hp[hp.attack_type.isin(["norm", "sqli"])].drop_duplicates("payload")
hp = pd.DataFrame({"Query": hp.payload.astype(str), "Label": (hp.attack_type == "sqli").astype(int)})
seen = set(pd.concat([train.Query, val.Query]).map(key))
hp_eval = hp[~hp.Query.map(key).isin(seen)].reset_index(drop=True)
print(f"HttpParams sızmasız: {len(hp_eval)}")

libinjection 4.0.1


HttpParams sızmasız: 23720


In [2]:
def vec(fn):
    return lambda xs: np.array([fn(x) for x in xs], dtype=int)

ml = {n: joblib.load(MODELS / f"{f}.joblib") for n, f in
      [("LR baza", "lr_v2"), ("LR + normallaşdırma", "lr_norm_v2"), ("LR + norm. + augmentation", "lr_norm_aug_v2")]}
DETECTORS = {
    "Qara siyahı (sanitizeInput)": vec(lambda q: flags(q, sanitize_input)),
    "Qara siyahı (sanitizeSearch)": vec(lambda q: flags(q, sanitize_search)),
    "libinjection 4.0.1": vec(libinj.is_sqli),
    **{n: (lambda m: lambda xs: m.predict(list(xs)).astype(int))(m) for n, m in ml.items()},
}

SQL_STMT_RE = r"^\s*\(?\s*(?:select|insert|update|delete|create|drop|alter|with|exec|truncate|grant)\b"
plain_benign = ((test.Label == 0) & ~test.Query.str.contains(SQL_STMT_RE, case=False, regex=True)).to_numpy()
print(f"adi giriş (SQL ifadəsi olmayan təmiz): {plain_benign.sum()}")

def rates(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return p[y == 1].mean(), p[y == 0].mean()

adi giriş (SQL ifadəsi olmayan təmiz): 1120


## 1. Orijinal test, obfuskasiya (A və B) və HttpParams

In [3]:
rows = []
for name, det in DETECTORS.items():
    p = det(test.Query)
    rec, fpr = rates(test.Label, p)
    row = {"detector": name, "recall": rec, "fpr": fpr, "fpr_plain_input": p[plain_benign].mean(),
           "f1": f1_score(test.Label, p)}
    for sc in ("A", "B"):
        sub = obf_long[obf_long.scenario == sc]
        uniq = pd.unique(sub.model_input)
        pred = dict(zip(uniq, det(uniq)))
        for t in TECH_ORDER:
            row[f"recall_{sc}_{t}"] = sub[sub.technique == t].model_input.map(pred).mean()
    worst = min(VALID_TECHS, key=lambda t: row[f"recall_A_{t}"])
    row["worst_technique_A"], row["worst_recall_A"] = worst, row[f"recall_A_{worst}"]
    ph = det(hp_eval.Query)
    row["httpparams_recall"], row["httpparams_fpr"] = rates(hp_eval.Label, ph)
    rows.append(row)
det_table = pd.DataFrame(rows).set_index("detector")
det_table[["recall", "fpr", "fpr_plain_input", "f1", "worst_technique_A", "worst_recall_A",
           "httpparams_recall", "httpparams_fpr"]].round(4)

,recall,fpr,fpr_plain_input,f1,worst_technique_A,worst_recall_A,httpparams_recall,httpparams_fpr
detector,,,,,,,,
Qara siyahı (sanitizeInput),0.9759,0.6485,0.1232,0.6475,double_url_encode,0.9138,0.9949,0.0868
Qara siyahı (sanitizeSearch),0.9496,0.5991,0.0179,0.6529,double_url_encode,0.7126,0.9639,0.0023
libinjection 4.0.1,0.9054,0.3876,0.0000,0.7169,double_url_encode,0.0000,0.9679,0.0000
LR baza,0.9960,0.0011,0.0018,0.9971,alt_whitespace,0.7777,0.9725,0.0003
LR + normallaşdırma,0.9948,0.0014,0.0018,0.9963,url_encode,0.9977,0.9998,0.0003
LR + norm. + augmentation,0.9937,0.0007,0.0009,0.9963,url_encode,0.9966,0.9998,0.0003


In [4]:
tech_tab = det_table[[f"recall_A_{t}" for t in TECH_ORDER]].T.mul(100).round(1)
tech_tab.index = [i.replace("recall_A_", "") for i in tech_tab.index]
tech_tab

detector,Qara siyahı (sanitizeInput),Qara siyahı (sanitizeSearch),libinjection 4.0.1,LR baza,LR + normallaşdırma,LR + norm. + augmentation
random_case,98.1,95.4,91.4,100.0,99.9,99.8
inline_comment_space,100.0,100.0,91.2,100.0,99.9,99.8
alt_whitespace,98.0,95.4,91.1,77.8,99.9,99.8
alt_whitespace_url,98.0,95.4,91.2,77.8,99.9,99.8
url_encode,97.9,95.2,90.8,99.9,99.8,99.7
double_url_encode,91.4,71.3,0.0,98.4,99.8,99.7
versioned_comment,100.0,100.0,99.4,100.0,99.9,99.8
logical_operator,96.4,96.0,96.1,100.0,100.0,100.0
hex_strings,100.0,99.0,93.6,99.8,100.0,99.8
keyword_split_comment,100.0,100.0,9.8,100.0,99.6,99.8


## 2. MySQL-də effektiv formalar (06 / 09 ilə eyni hovuz)
Effektiv forma hovuzu = orijinal + etibarlı obfuskasiyalar, MySQL-də effektiv olanlar (996 forma, 114 payload). Detektor "təmiz" deyirsə, forma keçmiş sayılır. Qara siyahı üçün 06-da ölçülmüş `survives_*` istifadə olunur. 95% CI — payload üzrə cluster bootstrap (1 000 təkrar, seed 42).

In [5]:
V = pd.read_csv(PROC / "06_mysql_effectiveness.csv", keep_default_na=False)
VALID = [t for t in V.variant.unique() if t != "keyword_split_comment"]
pool = V[V.variant.isin(["original"] + VALID) & V.effective].copy()
for name, det in DETECTORS.items():
    if name.startswith("Qara siyahı"):
        pool[f"pass::{name}"] = pool["survives_input" if "Input" in name else "survives_search"].astype(bool)
    else:
        pool[f"pass::{name}"] = det(pool.text) == 0

uniq_pids = np.unique(pool.payload_id)
rng = np.random.default_rng(SEED)
boot = [rng.choice(uniq_pids, size=len(uniq_pids), replace=True) for _ in range(1000)]
grp = pool.groupby("payload_id")
rows = []
for name in DETECTORS:
    col = f"pass::{name}"
    k_by, n_by = grp[col].sum(), grp.size()
    bvals = [k_by.loc[b].sum() / n_by.loc[b].sum() for b in boot]
    lo, hi = np.percentile(bvals, [2.5, 97.5])
    k = int(pool[col].sum())
    rows.append({"detector": name, "effective_forms": len(pool), "passed_forms": k,
                 "passed_unique_payloads": pool.loc[pool[col], "payload_id"].nunique(),
                 "passed_pct": 100 * k / len(pool), "ci_low": 100 * lo, "ci_high": 100 * hi})
bypass = pd.DataFrame(rows).set_index("detector")
bypass.round(2)

,effective_forms,passed_forms,passed_unique_payloads,passed_pct,ci_low,ci_high
detector,,,,,,
Qara siyahı (sanitizeInput),996,30,20,3.01,1.43,4.99
Qara siyahı (sanitizeSearch),996,123,22,12.35,7.88,17.74
libinjection 4.0.1,996,2,2,0.20,0.00,0.51
LR baza,996,46,23,4.62,2.92,6.43
LR + normallaşdırma,996,1,1,0.10,0.00,0.31
LR + norm. + augmentation,996,1,1,0.10,0.00,0.31


In [6]:
out = det_table.join(bypass)
out.to_csv(RES / "11_detector_comparison.csv")
out[["recall", "fpr", "fpr_plain_input", "worst_technique_A", "worst_recall_A", "httpparams_recall",
     "httpparams_fpr", "passed_forms", "passed_unique_payloads", "passed_pct"]].round(4)

,recall,fpr,fpr_plain_input,worst_technique_A,worst_recall_A,httpparams_recall,httpparams_fpr,passed_forms,passed_unique_payloads,passed_pct
detector,,,,,,,,,,
Qara siyahı (sanitizeInput),0.9759,0.6485,0.1232,double_url_encode,0.9138,0.9949,0.0868,30,20,3.0120
Qara siyahı (sanitizeSearch),0.9496,0.5991,0.0179,double_url_encode,0.7126,0.9639,0.0023,123,22,12.3494
libinjection 4.0.1,0.9054,0.3876,0.0000,double_url_encode,0.0000,0.9679,0.0000,2,2,0.2008
LR baza,0.9960,0.0011,0.0018,alt_whitespace,0.7777,0.9725,0.0003,46,23,4.6185
LR + normallaşdırma,0.9948,0.0014,0.0018,url_encode,0.9977,0.9998,0.0003,1,1,0.1004
LR + norm. + augmentation,0.9937,0.0007,0.0009,url_encode,0.9966,0.9998,0.0003,1,1,0.1004


## 3. libinjection-ın səhvləri
Nəyi buraxır (fingerprint ilə) və nəyi səhvən bloklayır.

In [7]:
fp_rows = []
for q, y in zip(test.Query, test.Label):
    r, fp = libinj.sqli(q)
    fp_rows.append((q, y, r, fp))
li = pd.DataFrame(fp_rows, columns=["Query", "Label", "result", "fingerprint"])
print("Buraxılan zərərli (nümunə):")
for q in li[(li.Label == 1) & (li.result != 1)].Query.sample(10, random_state=SEED):
    print("  ", repr(q[:110]))
print("\nSəhvən bloklanan təmiz (nümunə):")
for q, f in li[(li.Label == 0) & (li.result == 1)][["Query", "fingerprint"]].sample(10, random_state=SEED).values:
    print(f"   [{f}] {q[:100]!r}")
miss_pass = pool[pool["pass::libinjection 4.0.1"]]
print("\nMySQL-də effektiv, libinjection-dan keçən formalar — variant üzrə:")
print(miss_pass.variant.value_counts().to_string())
li.groupby("Label").result.value_counts().unstack().to_csv(RES / "11_libinjection_result_counts.csv")

Buraxılan zərərli (nümunə):
   'select * from generate_series  (  6730,6730,case when   (  6730  =  9431  )   then 1 else 0 end  )   limit 1--'
   "call regexp_substring  (  repeat  (  right  (  char  (  3702  )  ,0  )  ,500000000  )  ,null  )   and 'pczr'  "
   "1   )    )     )   waitfor delay '0:0:5' and    (    (     (  3481  =  3481"
   '1 where 8393  =  8393'
   '1"  )   where 4951  =  4951'
   'declare @q nvarchar   (  200  )   0x730065006c00650063007400200040004000760065007200730069006f006e00 exec  (  '
   "1\\'1"
   "waitfor delay '0:0:5'-- thuo"
   '1  )   where 2170  =  2170'
   "1' where 2967  =  2967"

Səhvən bloklanan təmiz (nümunə):
   [Eoknk] "SELECT * FROM Customers  WHERE Country = 'Germany' OR Country = 'Spain';"
   [Ef(n)] 'SELECT COUNT ( fliesID ) FROM grandfather'
   [Eoknk] "SELECT * FROM clothes WHERE parent = 'direction'  AND  ( pencil = 'heavy' OR matter = 'parts' ) "
   [Eoknk] "SELECT * FROM doubt WHERE fallen BETWEEN 'gradually' AND 'salmon'"
   [Eoknk] 'SE